# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    for attempt in range(tries):
        r = requests.get(BASE, params=params, timeout=10)

        if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])

        if r.status_code not in RETRY_STATUSES:
            r.raise_for_status()

        time.sleep(attempt + 1)

    raise RuntimeError('all tries failed')

Explanation: I wrote a loop that tries the request up to three times with a timeout, returns the daily block as a DataFrame on a 200, raises on an error such as 400 or 404, and waits a bit longer before each retry on a 429 or 500+.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
dates = ['2026-09-05', '2026-10-03']

used_fallback = False

try:
    raw = get_weather('2026-09-05', '2026-10-03')
    wx = raw[raw['time'].isin(dates)].copy()
except Exception as e:
    print('the API call failed:', e)
    used_fallback = True

print('used_fallback:', used_fallback)

used_fallback: False


Explanation: I made one request covering the whole range from September 5 to October 3 and then filtered down to just the two dates, and I wrapped it in a try so that a failed call sets used_fallback to True instead of stopping the notebook.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'

print('source:', wx['source'].iloc[0])
wx

source: open-meteo


,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


Explanation: I built wx from the sample data if the API call failed and from the real data if it worked, and I added a source column so each row says where it came from.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

wx['date'] = pd.to_datetime(wx['date'])

wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


Explanation: I renamed the API columns to my own names and turned the date text into a datetime so it can join to other tables.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]

wetter_date = wetter['date'].date()
wetter_rain = wetter['precip_mm']
drier_date = drier['date'].date()
drier_rain = drier['precip_mm']
gap = round(wetter_rain - drier_rain, 1)

print(wetter_date, 'was the wetter day with', wetter_rain, 'mm of rain, which is', gap, 'mm more than', drier_date, 'which got', drier_rain, 'mm.')

2026-10-03 was the wetter day with 14.7 mm of rain, which is 12.2 mm more than 2026-09-05 which got 2.5 mm.


Explanation: I used idxmax and idxmin to pull the wettest and driest of the two days, stored each date and rainfall in its own variable, then printed a sentence using those variables.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
from datetime import datetime

coordinates = '38.03, -78.51'
date_range = '2026-09-05 to 2026-10-03'
pulled_at = datetime.now().strftime('%Y-%m-%d %H:%M')
data_source = wx['source'].iloc[0]

print('endpoint:', BASE)
print('coordinates:', coordinates)
print('date range:', date_range)
print('pulled at:', pulled_at)
print('data source:', data_source)

endpoint: https://archive-api.open-meteo.com/v1/archive
coordinates: 38.03, -78.51
date range: 2026-09-05 to 2026-10-03
pulled at: 2026-10-12 00:03
data source: open-meteo


Explanation: I printed the endpoint, coordinates, date range, the time I ran the pull, and whether the data is real or the fallback sample.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame instead of raising an error, the merge onto 200 days of sales would still work and still give back 200 rows, so it would not look broken. Every sales row would just have blank weather columns because there was nothing to match against. The chart would then come out empty, or if the blanks got filled with zeros somewhere, it would look like every day had no rain and revenue was unrelated to it. Nobody would catch it because there is no error to catch since the notebook would run fine from top to bottom and the row count would appear to be right. As such, the function should raise instead, so the failure shows up in the console where people can fix it rather than in the chart where it can be mistakenly reported.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
orl_raw = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orl = orl_raw[orl_raw['time'].isin(dates)].copy()
orl['source'] = 'open-meteo'

orl = orl.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
orl['date'] = pd.to_datetime(orl['date'])
orl = orl[['date', 'temp_max', 'precip_mm', 'source']]

wx['city'] = 'Charlottesville'
orl['city'] = 'Orlando'

both = pd.concat([wx, orl], ignore_index=True)
both = both[['city', 'date', 'temp_max', 'precip_mm', 'source']]

both

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo


Explanation: I called the same function with Orlando's coordinates, cleaned it the same way, added a city column to each frame, and stacked them so there is one row per city per date.